## 11 - Pseudo RAG with Local LLM (Ollama)

This notebook demonstrates a simple retrieval-augmented generation (RAG) pipeline using:

- Local LLM (Llama 3.2 via Ollama)
- Keyword-based retrieval (no embeddings)
- Knowledge base from local text files

Pipeline:
User Query → Keyword Match → Context Injection → LLM Response

Limitations:
- Exact keyword matching
- No semantic search

Future Improvements:
- Add embeddings (FAISS / Chroma)
- Chunk documents
- Improve retrieval ranking

In [1]:
import os
import glob
from pathlib import Path
import gradio as gr
from dotenv import load_dotenv
from openai import OpenAI

In [2]:
load_dotenv(override=True)
OLLAMA_BASE_URL = "http://localhost:11434/v1"
OLLAMA_API_KEY = "ollama"
MODEL = "llama3.2:latest"

In [3]:
client = OpenAI(
    base_url=OLLAMA_BASE_URL,
    api_key=OLLAMA_API_KEY
)

In [4]:
knowledge = {}
filenames = glob.glob("knowledge-base/employees/*")

for filename in filenames:
    name = Path(filename).stem.split(' ')[-1]
    with open(filename, "r", encoding="utf-8") as f:
        knowledge[name.lower()] = f.read()

knowledge["lancaster"]

"# Avery Lancaster\n\n## Summary\n- **Date of Birth**: March 15, 1985\n- **Job Title**: Co-Founder & Chief Executive Officer (CEO)\n- **Location**: San Francisco, California\n- **Current Salary**: $225,000  \n\n## Insurellm Career Progression\n- **2015 - Present**: Co-Founder & CEO  \n  Avery Lancaster co-founded Insurellm in 2015 and has since guided the company to its current position as a leading Insurance Tech provider. Avery is known for her innovative leadership strategies and risk management expertise that have catapulted the company into the mainstream insurance market.  \n\n- **2013 - 2015**: Senior Product Manager at Innovate Insurance Solutions  \n  Before launching Insurellm, Avery was a leading Senior Product Manager at Innovate Insurance Solutions, where she developed groundbreaking insurance products aimed at the tech sector.  \n\n- **2010 - 2013**: Business Analyst at Edge Analytics  \n  Prior to joining Innovate, Avery worked as a Business Analyst, focusing on market t

In [5]:
filenames = glob.glob("knowledge-base/products/*")

for filename in filenames:
    name = Path(filename).stem
    with open(filename, "r", encoding="utf-8") as f:
        knowledge[name.lower()] = f.read()

knowledge.keys()

dict_keys(['martinez', 'thomson', 'bishop', "o'brien", 'thompson', 'liu', 'harper', 'walker', 'anderson', 'greene', 'chen', 'rivera', 'foster', 'blake', 'adams', 'brooks', 'patel', 'kim', 'spencer', 'sharma', 'zhang', 'johnson', 'tran', 'wilson', 'williams', 'park', 'rodriguez', 'lancaster', 'trenton', 'carter', 'rellm', 'claimllm', 'carllm', 'lifellm', 'healthllm', 'homellm', 'bizllm', 'markellm'])

In [6]:
print("Loaded knowledge keys:")
print(list(knowledge.keys())[:20])

Loaded knowledge keys:
['martinez', 'thomson', 'bishop', "o'brien", 'thompson', 'liu', 'harper', 'walker', 'anderson', 'greene', 'chen', 'rivera', 'foster', 'blake', 'adams', 'brooks', 'patel', 'kim', 'spencer', 'sharma']


In [7]:
SYSTEM_PREFIX = """
You represent Insurellm, the Insurance Tech company.
You are an expert in answering questions about Insurellm, its employees, and its products.
You are provided with additional context that might be relevant to the user's question.
Give brief, accurate answers.
If the answer is not in the provided context, say you don't know.
Do not invent facts.

Relevant context:
"""

In [8]:
def get_relevant_context(message):
    text = ''.join(ch for ch in message if ch.isalpha() or ch.isspace())
    words = text.lower().split()
    return [knowledge[word] for word in words if word in knowledge]

get_relevant_context("Who is lancaster?")
get_relevant_context("Who is Lancaster and what is carllm?")

["# Avery Lancaster\n\n## Summary\n- **Date of Birth**: March 15, 1985\n- **Job Title**: Co-Founder & Chief Executive Officer (CEO)\n- **Location**: San Francisco, California\n- **Current Salary**: $225,000  \n\n## Insurellm Career Progression\n- **2015 - Present**: Co-Founder & CEO  \n  Avery Lancaster co-founded Insurellm in 2015 and has since guided the company to its current position as a leading Insurance Tech provider. Avery is known for her innovative leadership strategies and risk management expertise that have catapulted the company into the mainstream insurance market.  \n\n- **2013 - 2015**: Senior Product Manager at Innovate Insurance Solutions  \n  Before launching Insurellm, Avery was a leading Senior Product Manager at Innovate Insurance Solutions, where she developed groundbreaking insurance products aimed at the tech sector.  \n\n- **2010 - 2013**: Business Analyst at Edge Analytics  \n  Prior to joining Innovate, Avery worked as a Business Analyst, focusing on market 

In [9]:
def additional_context(message):
    relevant_context = get_relevant_context(message)
    if not relevant_context:
        result = "There is no additional context relevant to the user's question."
    else:
        result = "The following additional context might be relevant in answering the user's question:\n\n"
        result += "\n\n".join(relevant_context)
    return result

In [10]:
def chat(message, history):
    system_message = SYSTEM_PREFIX + additional_context(message)

    messages = [{"role": "system", "content": system_message}]

    # Gradio type="messages" history format
    for item in history:
        if isinstance(item, dict) and "role" in item and "content" in item:
            messages.append(item)

    messages.append({"role": "user", "content": message})

    response = client.chat.completions.create(
        model=MODEL,
        messages=messages,
        temperature=0.2
    )

    return response.choices[0].message.content


In [11]:
gr.ChatInterface(fn=chat).launch(inbrowser=True)

* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.
